# Assignment 2 - Walk-Forward Models

This notebook runs the dual-path forecasting pipeline: direct bus-level forecasting and zone forecast plus historical bus-share allocation.

## Strategy

The first executed fold is the prototype `2022 Jan-Mar -> 2022 Apr` fold. The output table also records the expanded 2022, multi-year, and final 2025 holdout folds for later scaling.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / "data").exists() and (candidate / "solution" / "assignment_2" / "src").exists():
        repo_root = candidate
        break
else:
    raise FileNotFoundError("Could not find ECESIS repository root from notebook path.")

src_dir = repo_root / "solution" / "assignment_2" / "src"
outputs_dir = repo_root / "solution" / "assignment_2" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(src_dir))

pd.set_option("display.max_columns", None)


In [ ]:
from pipeline import run_prototype_pipeline

results = run_prototype_pipeline(outputs_dir, n_buses=20)
{k: v.shape for k, v in results.items()}

## Direct Bus-Level Forecast

A single global HistGradientBoostingRegressor predicts hourly bus-level `pd` using calendar, categorical bus/zone, lag, rolling, historical-average, and zone-context features.

In [ ]:
results["bus_forecast_direct"].head()

## Zone Forecast + Bus-Share Allocation

The hierarchical path aggregates bus `pd` to zone-hour, forecasts zone load, then allocates predicted zone load back to buses using training-window historical bus shares conditioned by zone, bus, HE, and day-of-week.

In [ ]:
results["bus_forecast_zone_allocated"].head()

## Output Files

The CSV outputs in `assignment_2/outputs/` are intended deliverables for model comparison. Re-running this notebook intentionally refreshes those deliverables for the configured fold.

In [ ]:
sorted(p.name for p in outputs_dir.glob("*.csv"))